# Q7: Modeling

**Phase 8:** Modeling  
**Points: 9 points**

**Focus:** Train multiple models, evaluate performance, compare models, extract feature importance.

**Lecture Reference:** Lecture 11, Notebook 4 ([`11/demo/04_modeling_results.ipynb`](https://github.com/christopherseaman/datasci_217/blob/main/11/demo/04_modeling_results.ipynb)), Phase 8. Also see Lecture 10 (modeling with sklearn and XGBoost).

---

## Setup

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import os

# Load prepared data from Q6
X_train = pd.read_csv('output/q6_X_train.csv')
X_test = pd.read_csv('output/q6_X_test.csv')
y_train = pd.read_csv('output/q6_y_train.csv').squeeze()  # Convert to Series
y_test = pd.read_csv('output/q6_y_test.csv').squeeze()

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

Training set: (157237, 88)
Test set: (39310, 88)


---

## Objective

Train multiple models, evaluate performance, compare models, and extract feature importance.

---

## ⚠️ Data Leakage Warning

If you see suspiciously perfect model performance, this likely indicates data leakage. Common warning signs:

**Warning Metrics:**
- **Perfect R² = 1.0000** (or very close, like 0.9999+)
- **Zero or near-zero RMSE/MAE** (e.g., RMSE < 0.01°C for temperature prediction)
- **Train and test performance nearly identical** (difference < 0.01)
- **Unrealistic precision**: Errors smaller than measurement precision (e.g., < 0.1°C for temperature sensors)
- **Feature correlation > 0.99** with target (check correlations between features and target)

**Common Causes:**
- **Circular prediction logic**: Using rolling windows of the target variable to predict itself
  - Example: Using `air_temp_rolling_7h` to predict `Air Temperature`
  - This is like predicting temperature from smoothed temperature - circular reasoning!
- **Features nearly identical to target**: Any feature with correlation > 0.99 with the target
- **Including target variable directly**: Accidentally including the target in features

**How to Check:**
- Calculate correlations between each feature and the target
- If any feature has correlation > 0.95, investigate whether it's legitimate or leakage
- For time series: Be especially careful with rolling windows, lag features, or any transformation of the target variable

**Example of Problematic Feature:**
- `air_temp_rolling_7h` (7-hour rolling mean of Air Temperature) when predicting Air Temperature
- This feature has ~99.4% correlation with the target - too high to be useful and indicates circular logic

**Solution:**
- Only create rolling windows for **predictor variables**, not the target
- Use rolling windows of: Wind Speed, Humidity, Barometric Pressure, etc.
- Avoid rolling windows of: Air Temperature (if that's your target)

---

## Required Artifacts

You must create exactly these 3 files in the `output/` directory:

### 1. `output/q7_predictions.csv`
**Format:** CSV file
**Required Columns (exact names):**
- `actual` - Actual target values from test set
- `predicted_linear` or `predicted_model1` - Predictions from first model (e.g., Linear Regression)
- `predicted_xgboost` or `predicted_model2` - Predictions from second model (e.g., XGBoost)
- Additional columns for additional models (e.g., `predicted_random_forest` or `predicted_model3`)

**Requirements:**
- Must have at least 2 model prediction columns (in addition to `actual`)
- All values must be numeric (float)
- Same number of rows as test set
- **No index column** (save with `index=False`)

**Example:**
```csv
actual,predicted_linear,predicted_xgboost
15.2,14.8,15.1
15.3,15.0,15.2
...
```

### 2. `output/q7_model_metrics.txt`
**Format:** Plain text file
**Content:** Performance metrics for each model
**Required information for each model:**
- Model name
- At least R² score for both train and test sets (additional metrics like RMSE, MAE recommended but optional)

**Requirements:**
- Clearly labeled (model name, metric name)
- **At minimum:** R² (or R-squared or R^2) for train and test for each model
- Additional metrics (RMSE, MAE) are recommended for a complete analysis
- Format should be readable

**Example format (minimum - R² only):**
```
MODEL PERFORMANCE METRICS
========================

LINEAR REGRESSION:
  Train R²: 0.3048
  Test R²:  0.3046

XGBOOST:
  Train R²: 0.9091
  Test R²:  0.7684
```

**Example format (recommended - with additional metrics):**
```
MODEL PERFORMANCE METRICS
========================

LINEAR REGRESSION:
  Train R²: 0.3048
  Test R²:  0.3046
  Train RMSE: 8.42
  Test RMSE:  8.43
  Train MAE:  7.03
  Test MAE:   7.04

XGBOOST:
  Train R²: 0.9091
  Test R²:  0.7684
  Train RMSE: 3.45
  Test RMSE:  4.87
  Train MAE:  2.58
  Test MAE:   3.66
```

### 3. `output/q7_feature_importance.csv`
**Format:** CSV file
**Required Columns (exact names):** `feature`, `importance`
**Content:** Feature importance from tree-based models (XGBoost, Random Forest)
**Requirements:**
- One row per feature
- `feature`: Feature name (string)
- `importance`: Importance score (float, typically 0-1, sum to 1)
- Sorted by importance (descending)
- **No index column** (save with `index=False`)

**Note:** Tree-based models (XGBoost, Random Forest) provide feature importance directly via `.feature_importances_`. If using only Linear Regression, you can use the absolute values of coefficients as a proxy for importance.

**Example:**
```csv
feature,importance
Air Temperature,0.6539
hour,0.1234
month,0.0892
Water Temperature,0.0456
...
```

---

## Requirements Checklist

- [ ] At least 2 different models trained
  - **Suggested:** Linear Regression and XGBoost (or Random Forest)
  - You may choose other models if appropriate
- [ ] Performance evaluated on both train and test sets
- [ ] Models compared
- [ ] Feature importance extracted
  - Tree-based models: use `.feature_importances_`
  - Linear Regression: use absolute coefficient values
- [ ] Model performance documented with **at least R²** (additional metrics like RMSE, MAE recommended)
- [ ] All 3 required artifacts saved with exact filenames

---

## Your Approach

1. **Check for data leakage** - Before training, compute correlations between features and target. Any feature with correlation > 0.95 should be investigated and considered for removal.
2. **Train at least 2 models** - Fit models to training data, generate predictions for both train and test sets
3. **Calculate metrics** - At minimum R² for train and test; RMSE and MAE recommended
4. **Extract feature importance** - Use `.feature_importances_` for tree-based models, or coefficient magnitudes for linear models
5. **Save predictions** - DataFrame with `actual` column plus `predicted_*` columns for each model
6. **Save metrics** - Write clearly labeled metrics to text file

---

## Decision Points

- **Model selection:** Train at least 2 different models. We suggest starting with **Linear Regression** and **XGBoost** - these work well and demonstrate different modeling approaches (linear vs gradient boosting). You may choose other models if appropriate (e.g., Random Forest, Gradient Boosting, etc.). See Lecture 11 Notebook 4 for examples.
- **Evaluation metrics:** Report at least one metric for each model. We suggest **R² score** (coefficient of determination) - it works for both Linear Regression and XGBoost, and all regression models. It measures the proportion of variance explained and is easy to interpret. Alternative metrics that work well for both models include **RMSE** (Root Mean Squared Error) or **MAE** (Mean Absolute Error). You may include additional metrics if relevant (e.g., MAPE, adjusted R²). Compare train vs test performance to check for overfitting.
- **Feature importance:** If using tree-based models (like XGBoost), extract feature importance to understand which features matter most.

---

## Interpreting Model Performance

**Warning Signs of Data Leakage:**
- R² = 1.0000 (perfect score) or R² > 0.999
- RMSE or MAE = 0.0 or unrealistically small (< 0.01 for temperature)
- Train and test performance nearly identical (difference < 0.01)
- Any feature with correlation > 0.99 with target

**Realistic Expectations:**
- For temperature prediction: RMSE of 0.5-2.0°C is realistic
- R² of 0.85-0.98 is strong but realistic
- Some difference between train and test performance is normal

**If you see warning signs:**
1. Check your features for data leakage (see Data Leakage Warning above)
2. Calculate correlations between features and target
3. Remove features that are transformations of the target variable
4. Re-train models and verify performance is now realistic

---

## Checkpoint

After Q7, you should have:
- [ ] At least 2 models trained (suggested: Linear Regression and XGBoost)
- [ ] Performance metrics calculated (at minimum: one metric like R², RMSE, or MAE for train and test; additional metrics recommended)
- [ ] Models compared
- [ ] Feature importance extracted (if applicable - tree-based models like XGBoost)
- [ ] All 3 artifacts saved: `q7_predictions.csv`, `q7_model_metrics.txt`, `q7_feature_importance.csv`

---

**Next:** Continue to `q8_results.md` for Results.


### 1. `output/q7_predictions.csv`
**Format:** CSV file
**Required Columns (exact names):**
- `actual` - Actual target values from test set
- `predicted_linear` or `predicted_model1` - Predictions from first model (e.g., Linear Regression)
- `predicted_xgboost` or `predicted_model2` - Predictions from second model (e.g., XGBoost)
- Additional columns for additional models (e.g., `predicted_random_forest` or `predicted_model3`)

**Requirements:**
- Must have at least 2 model prediction columns (in addition to `actual`)
- All values must be numeric (float)
- Same number of rows as test set
- **No index column** (save with `index=False`)

**Example:**
```csv
actual,predicted_linear,predicted_xgboost
15.2,14.8,15.1
15.3,15.0,15.2
...
```

In [2]:
# 7.1 Modeling Predictions CSV

print("="*80)
print("Q7.1: MODEL TRAINING AND PREDICTIONS CSV")
print("="*80)

# Load training/testing data
print("\n1. LOADING TRAIN/TEST DATA")
print("-" * 80)

X_train = pd.read_csv('output/q6_X_train.csv')
X_test = pd.read_csv('output/q6_X_test.csv')
y_train = pd.read_csv('output/q6_y_train.csv')
y_test = pd.read_csv('output/q6_y_test.csv')

print(f"  X_train: {X_train.shape}")
print(f"  X_test: {X_test.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  y_test: {y_test.shape}")

# Get target variable name
target_name = y_train.columns[0]
print(f"\n  Target variable: {target_name}")

# Convert to numpy arrays for sklearn
y_train_values = y_train[target_name].values
y_test_values = y_test[target_name].values

# DATA LEAKAGE CHECK
print("\n2. DATA LEAKAGE CHECK")
print("-" * 80)
print("!  Checking for potential data leakage...")

# Calculate correlations between features and target
feature_target_corr = pd.DataFrame({
    'feature': X_train.columns,
    'correlation': [X_train[col].corr(pd.Series(y_train_values)) for col in X_train.columns]
})
feature_target_corr['abs_correlation'] = feature_target_corr['correlation'].abs()
feature_target_corr = feature_target_corr.sort_values('abs_correlation', ascending=False)

# Check for unusual correlations
high_corr_features = feature_target_corr[feature_target_corr['abs_correlation'] > 0.95]

if len(high_corr_features) > 0:
    print(f"\n!  WARNING: Found {len(high_corr_features)} features with correlation > 0.95")
    print(f"These may indicate data leakage:\n")
    for _, row in high_corr_features.head(10).iterrows():
        print(f"  - {row['feature']}: {row['correlation']:.4f}")
    
    # Identify rolling/lag features of the target
    target_base = target_name.lower().replace(' ', '_').replace('_', '')
    leakage_features = []
    
    for feat in high_corr_features['feature']:
        feat_clean = feat.lower().replace(' ', '_').replace('_', '')
        # See if feature is a rolling/lag/derived version of target
        if target_base in feat_clean and any(x in feat.lower() for x in ['rolling', 'lag', 'mean', 'std', 'deviation']):
            leakage_features.append(feat)
    
    if leakage_features:
        print(f"\n!  CRITICAL: Detected likely data leakage features:")
        for feat in leakage_features:
            print(f"  - {feat} (appears to be derived from target variable)")
        print(f"\nRemoving {len(leakage_features)} leakage features...")
        
        # Remove leakage features
        X_train = X_train.drop(columns=leakage_features)
        X_test = X_test.drop(columns=leakage_features)
        
        print(f"  New shape: {X_train.shape}")
    else:
        print(f"\n  High correlations found but no obvious leakage features detected")
        print(f"  (These may be legitimate predictors)")
else:
    print(f"  No suspicious correlations detected (all correlations < 0.95)")

print(f"\nTop 10 feature-target correlations:")
for _, row in feature_target_corr.head(10).iterrows():
    print(f"  {row['feature']}: {row['correlation']:.3f}")

# Feature selection (use top features to reduce complexity)
print("\n FEATURE SELECTION")
print("-" * 80)

# Select top 50 features by absolute correlation
n_features = min(50, len(X_train.columns))
top_features = feature_target_corr.head(n_features)['feature'].tolist()

# Filter out leakage features that were removed
if 'leakage_features' in locals() and leakage_features:
    top_features = [f for f in top_features if f not in leakage_features]
    print(f"Filtered out {len(leakage_features)} leakage features from top features")

# Make sure we have enough features after filtering
n_features = min(50, len(top_features))
top_features = top_features[:n_features]

print(f"Selecting top {n_features} features by correlation with target")
X_train_selected = X_train[top_features].copy()
X_test_selected = X_test[top_features].copy()

print(f"  Selected features shape: {X_train_selected.shape}")

# Handle any remaining missing values
if X_train_selected.isnull().any().any() or X_test_selected.isnull().any().any():
    print(f"\n !  Handling missing values...")
    # Fill with column means
    for col in X_train_selected.columns:
        if X_train_selected[col].isnull().any():
            mean_val = X_train_selected[col].mean()
            X_train_selected[col].fillna(mean_val, inplace=True)
            X_test_selected[col].fillna(mean_val, inplace=True)
    print(f"  Missing values filled")

# Train multiple models
print("\n TRAINING MULTIPLE MODELS")
print("-" * 80)

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
}

predictions = {}
trained_models = {}

for model_name, model in models.items():
    print(f"\nTraining {model_name}...")
    
    # Train model
    model.fit(X_train_selected, y_train_values)
    
    # Make predictions
    train_pred = model.predict(X_train_selected)
    test_pred = model.predict(X_test_selected)
    
    # Store predictions
    predictions[model_name] = {
        'train': train_pred,
        'test': test_pred
    }
    trained_models[model_name] = model
    
    # Calculate metrics
    train_r2 = r2_score(y_train_values, train_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train_values, train_pred))
    train_mae = mean_absolute_error(y_train_values, train_pred)
    
    test_r2 = r2_score(y_test_values, test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test_values, test_pred))
    test_mae = mean_absolute_error(y_test_values, test_pred)
    
    print(f"  Training:   R²={train_r2:.4f}, RMSE={train_rmse:.4f}, MAE={train_mae:.4f}")
    print(f"  Test:       R²={test_r2:.4f}, RMSE={test_rmse:.4f}, MAE={test_mae:.4f}")
    
    # Check for data leakage indicators
    if test_r2 > 0.999 or test_rmse < 0.01:
        print(f"  !  WARNING: Suspiciously high performance - possible data leakage!")
    elif abs(train_r2 - test_r2) < 0.01 and test_r2 > 0.95:
        print(f"  !  WARNING: Train/test performance too similar - check for leakage!")

print(f"\n  All models trained successfully")

# Create predictions dataframe
print("\n  CREATING PREDICTIONS DATAFRAME")
print("-" * 80)

predictions_df = pd.DataFrame({
    'actual': y_test_values
})

# Add predictions from each model with clean column names
model_name_mapping = {
    'Linear Regression': 'predicted_linear',
    'Decision Tree': 'predicted_tree',
    'Random Forest': 'predicted_random_forest'
}

for model_name, col_name in model_name_mapping.items():
    if model_name in predictions:
        predictions_df[col_name] = predictions[model_name]['test']

print(f"  Predictions dataframe created")
print(f"  Shape: {predictions_df.shape}")
print(f"  Columns: {list(predictions_df.columns)}")

# Display sample
print(f"\nSample predictions:")
print(predictions_df.head(10))

# Save predictions
print("\n6. SAVING PREDICTIONS")
print("-" * 80)

predictions_df.to_csv('output/q7_predictions.csv', index=False)
print(f"  Saved to: output/q7_predictions.csv")
print(f"  Shape: {predictions_df.shape[0]:,} rows × {predictions_df.shape[1]} columns")
print(f"  Format: actual + {predictions_df.shape[1]-1} model predictions")



# Summary
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"  Trained {len(models)} models:")
for name in models.keys():
    print(f"  - {name}")
print(f"\n  Generated predictions for test set:")
print(f"  - Test samples: {len(predictions_df):,}")
print(f"  - Models: {len([c for c in predictions_df.columns if 'predicted' in c])}")
print(f"\n  Saved to: output/q7_predictions.csv")
print(f"  Data leakage check performed")
print(f"  Feature selection applied ({n_features} features)")

print("\n" + "="*80)
print("MODEL TRAINING AND PREDICTIONS COMPLETE")
print("="*80)

Q7.1: MODEL TRAINING AND PREDICTIONS CSV

1. LOADING TRAIN/TEST DATA
--------------------------------------------------------------------------------
  X_train: (157237, 88)
  X_test: (39310, 88)
  y_train: (157237, 1)
  y_test: (39310, 1)

  Target variable: Air Temperature

2. DATA LEAKAGE CHECK
--------------------------------------------------------------------------------
!  Checking for potential data leakage...

!  WARNING: Found 1 features with correlation > 0.95
These may indicate data leakage:

  - Wet Bulb Temperature: 0.9785

  High correlations found but no obvious leakage features detected
  (These may be legitimate predictors)

Top 10 feature-target correlations:
  Wet Bulb Temperature: 0.978
  Wet Bulb Temperature_lag_24h: 0.933
  Air Temperature_lag_24h: 0.932
  Air Temperature_rolling_mean_168h: 0.916
  Wet Bulb Temperature_rolling_mean_168h: 0.907
  Solar Radiation_rolling_std_168h: 0.648
  Solar Radiation_rolling_mean_168h: 0.616
  Wind Speed_rolling_mean_168h: -0.5

/Users/eekod/Library/Mobile Documents/com~apple~CloudDocs/UCSF MAS/DATASCI 217 Python/DATSCI217 Final/ds217-11-final-cshiu-1/venv/lib/python3.9/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/eekod/Library/Mobile Documents/com~apple~CloudDocs/UCSF MAS/DATASCI 217 Python/DATSCI217 Final/ds217-11-final-cshiu-1/venv/lib/python3.9/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/eekod/Library/Mobile Documents/com~apple~CloudDocs/UCSF MAS/DATASCI 217 Python/DATSCI217 Final/ds217-11-final-cshiu-1/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/eekod/Library/Mobile Documents/com~apple~CloudDocs/UCSF MAS/DATASCI 217 Python/DATSCI217 Final/ds217-11-final-cshiu-1/venv/lib/python3.9/site-packages/sklearn/linear_

  Training:   R²=1.0000, RMSE=0.0000, MAE=0.0000
  Test:       R²=1.0000, RMSE=0.0000, MAE=0.0000
  !  WARNING: Suspiciously high performance - possible data leakage!

Training Random Forest...
  Training:   R²=0.9969, RMSE=0.5888, MAE=0.3961
  Test:       R²=0.9909, RMSE=0.9708, MAE=0.5179
  !  WARNING: Train/test performance too similar - check for leakage!

  All models trained successfully

  CREATING PREDICTIONS DATAFRAME
--------------------------------------------------------------------------------
  Predictions dataframe created
  Shape: (39310, 3)
  Columns: ['actual', 'predicted_linear', 'predicted_random_forest']

Sample predictions:
   actual  predicted_linear  predicted_random_forest
0   28.89             28.89                28.943548
1   29.60             29.60                29.461413
2   29.06             29.06                28.784318
3   29.50             29.50                29.323730
4   29.06             29.06                28.651696
5   29.50             29.50 

In [3]:
# 7.2 Model Metrics Report txt

print("="*80)
print("Q7.2: GENERATING MODEL METRICS REPORT")
print("="*80)

# Load train/test data
print("\n LOADING DATA AND PREDICTIONS")
print("-" * 80)

X_train = pd.read_csv('output/q6_X_train.csv')
X_test = pd.read_csv('output/q6_X_test.csv')
y_train = pd.read_csv('output/q6_y_train.csv')
y_test = pd.read_csv('output/q6_y_test.csv')
predictions_df = pd.read_csv('output/q7_predictions.csv')

print(f" Loaded all datasets")

# Get target variable name
target_name = y_train.columns[0]
y_train_values = y_train[target_name].values
y_test_values = y_test[target_name].values

print(f" Target variable: {target_name}")

# DATA LEAKAGE CHECK AND FEATURE SELECTION (get from training script)
print("\n  PREPARING FEATURES")
print("-" * 80)

# Calculate correlations for leakage detection
feature_target_corr = pd.DataFrame({
    'feature': X_train.columns,
    'correlation': [X_train[col].corr(pd.Series(y_train_values)) for col in X_train.columns]
})
feature_target_corr['abs_correlation'] = feature_target_corr['correlation'].abs()
feature_target_corr = feature_target_corr.sort_values('abs_correlation', ascending=False)

# Remove leakage features
high_corr_features = feature_target_corr[feature_target_corr['abs_correlation'] > 0.95]
target_base = target_name.lower().replace(' ', '_').replace('_', '')
leakage_features = []

for feat in high_corr_features['feature']:
    feat_clean = feat.lower().replace(' ', '_').replace('_', '')
    if target_base in feat_clean and any(x in feat.lower() for x in ['rolling', 'lag', 'mean', 'std', 'deviation']):
        leakage_features.append(feat)

if leakage_features:
    X_train = X_train.drop(columns=leakage_features)
    X_test = X_test.drop(columns=leakage_features)
    print(f"Removed {len(leakage_features)} leakage features")

# Select top features
n_features = min(50, len(X_train.columns))
top_features = feature_target_corr.head(n_features)['feature'].tolist()
top_features = [f for f in top_features if f not in leakage_features]

X_train_selected = X_train[top_features].copy()
X_test_selected = X_test[top_features].copy()

# Handle missing values
for col in X_train_selected.columns:
    if X_train_selected[col].isnull().any():
        mean_val = X_train_selected[col].mean()
        X_train_selected[col].fillna(mean_val, inplace=True)
        X_test_selected[col].fillna(mean_val, inplace=True)

print(f"  Using {len(top_features)} features for modeling")

# Train models and collect metrics
print("\n TRAINING MODELS AND CALCULATING METRICS")
print("-" * 80)

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
}

metrics_data = {}

for model_name, model in models.items():
    print(f"  Training {model_name}...")
    
    # Train model
    model.fit(X_train_selected, y_train_values)
    
    # Make predictions
    train_pred = model.predict(X_train_selected)
    test_pred = model.predict(X_test_selected)
    
    # Calculate metrics
    metrics_data[model_name] = {
        'train_r2': r2_score(y_train_values, train_pred),
        'test_r2': r2_score(y_test_values, test_pred),
        'train_rmse': np.sqrt(mean_squared_error(y_train_values, train_pred)),
        'test_rmse': np.sqrt(mean_squared_error(y_test_values, test_pred)),
        'train_mae': mean_absolute_error(y_train_values, train_pred),
        'test_mae': mean_absolute_error(y_test_values, test_pred)
    }

print(f"  All models trained and evaluated")

# Generate report
print("\n GENERATING METRICS REPORT")
print("-" * 80)

report_lines = []

def add_line(text="", indent=0):
    """Helper function to add formatted lines"""
    report_lines.append("  " * indent + text)

# Report header
add_line("MODEL PERFORMANCE METRICS")
add_line("=" * 70)
add_line()
add_line(f"Target Variable: {target_name}")
add_line(f"Training Samples: {len(X_train_selected):,}")
add_line(f"Test Samples: {len(X_test_selected):,}")
add_line(f"Number of Features: {len(top_features)}")
add_line()
add_line("=" * 70)
add_line()

# Metrics for each model
for model_name, metrics in metrics_data.items():
    add_line(model_name.upper() + ":")
    add_line("-" * 70)
    
    # R² scores (required)
    add_line(f"Train R²: {metrics['train_r2']:.4f}")
    add_line(f"Test R²:  {metrics['test_r2']:.4f}")
    
    # RMSE (recommended)
    add_line(f"Train RMSE: {metrics['train_rmse']:.4f}")
    add_line(f"Test RMSE:  {metrics['test_rmse']:.4f}")
    
    # MAE (recommended)
    add_line(f"Train MAE: {metrics['train_mae']:.4f}")
    add_line(f"Test MAE:  {metrics['test_mae']:.4f}")
    
    # Overfitting check
    r2_diff = metrics['train_r2'] - metrics['test_r2']
    if r2_diff > 0.1:
        add_line(f"!  Overfitting detected (R² difference: {r2_diff:.4f})")
    
    # Data leakage check
    if metrics['test_r2'] > 0.999 or metrics['test_rmse'] < 0.01:
        add_line(f"!  WARNING: Suspiciously high performance - check for data leakage!")
    
    add_line()

# Model comparison
add_line("=" * 70)
add_line("MODEL COMPARISON")
add_line("=" * 70)
add_line()

# Find best model by test R²
best_model = max(metrics_data.items(), key=lambda x: x[1]['test_r2'])
worst_model = min(metrics_data.items(), key=lambda x: x[1]['test_r2'])

add_line(f"Best Model (by Test R²): {best_model[0]}")
add_line(f"  Test R²: {best_model[1]['test_r2']:.4f}")
add_line(f"  Test RMSE: {best_model[1]['test_rmse']:.4f}")
add_line(f"  Test MAE: {best_model[1]['test_mae']:.4f}")
add_line()

add_line(f"Simplest Model: Linear Regression")
if 'Linear Regression' in metrics_data:
    lr_metrics = metrics_data['Linear Regression']
    add_line(f"  Test R²: {lr_metrics['test_r2']:.4f}")
    add_line(f"  Test RMSE: {lr_metrics['test_rmse']:.4f}")
add_line()

# Ranking by Test R²
add_line("Ranking by Test R² (Best to Worst):")
sorted_models = sorted(metrics_data.items(), key=lambda x: x[1]['test_r2'], reverse=True)
for i, (name, metrics) in enumerate(sorted_models, 1):
    add_line(f"  {i}. {name}: R²={metrics['test_r2']:.4f}, RMSE={metrics['test_rmse']:.4f}")

add_line()

# Performance analysis
add_line("=" * 70)
add_line("PERFORMANCE ANALYSIS")
add_line("=" * 70)
add_line()

# Calculate average metrics
avg_train_r2 = np.mean([m['train_r2'] for m in metrics_data.values()])
avg_test_r2 = np.mean([m['test_r2'] for m in metrics_data.values()])
avg_test_rmse = np.mean([m['test_rmse'] for m in metrics_data.values()])

add_line(f"Average Test R²: {avg_test_r2:.4f}")
add_line(f"Average Test RMSE: {avg_test_rmse:.4f}")
add_line()

# Interpretation
if avg_test_r2 > 0.8:
    add_line("Overall Performance: EXCELLENT")
    add_line("  Models show strong predictive power")
elif avg_test_r2 > 0.6:
    add_line("Overall Performance: GOOD")
    add_line("  Models capture most of the variance in the target")
elif avg_test_r2 > 0.4:
    add_line("Overall Performance: MODERATE")
    add_line("  Models show reasonable predictive ability")
else:
    add_line("Overall Performance: POOR")
    add_line("  Models struggle to predict the target variable")
    add_line("  Consider: more features, feature engineering, or different target")

add_line()

# Overfitting analysis
overfit_count = sum(1 for m in metrics_data.values() if m['train_r2'] - m['test_r2'] > 0.1)
if overfit_count > 0:
    add_line(f"!  {overfit_count} model(s) show signs of overfitting")
    add_line("  Consider: regularization, simpler models, or more training data")
else:
    add_line("! No significant overfitting detected across models")

add_line()

# Footer
add_line("=" * 70)
add_line("Note: R² (R-squared) measures proportion of variance explained")
add_line("      RMSE (Root Mean Squared Error) in target variable units")
add_line("      MAE (Mean Absolute Error) in target variable units")
add_line("=" * 70)

# Write report to file
report_content = "\n".join(report_lines)

with open('output/q7_model_metrics.txt', 'w') as f:
    f.write(report_content)

print(f"  Saved to: output/q7_model_metrics.txt")
print(f"  Lines: {len(report_lines)}")

with open('output/q7_model_metrics.txt', 'r') as f:
    saved_content = f.read()

print(f"  File saved successfully")
print(f"  Contains model names: {all(name.upper() in saved_content for name in models.keys())}")
print(f"  Contains R² scores: {'R²' in saved_content or 'R-squared' in saved_content}")
print(f"  Contains RMSE: {'RMSE' in saved_content}")
print(f"  Contains MAE: {'MAE' in saved_content}")
print(f"  Contains train/test metrics: {'Train' in saved_content and 'Test' in saved_content}")

# Summary
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"  Generated metrics for {len(models)} models")
print(f"  Best model: {best_model[0]} (Test R²: {best_model[1]['test_r2']:.4f})")
print(f"  Metrics included:")
print(f"  - R² (R-squared) - REQUIRED ✓")
print(f"  - RMSE (Root Mean Squared Error) - RECOMMENDED ✓")
print(f"  - MAE (Mean Absolute Error) - RECOMMENDED ✓")
print(f"\n  Report includes:")
print(f"  - Individual model performance")
print(f"  - Model comparison and ranking")
print(f"  - Performance analysis")
print(f"  - Overfitting detection")
print(f"\n  Saved to: output/q7_model_metrics.txt")

print("\n" + "="*80)
print("MODEL METRICS REPORT COMPLETE")
print("="*80)

Q7.2: GENERATING MODEL METRICS REPORT

 LOADING DATA AND PREDICTIONS
--------------------------------------------------------------------------------
 Loaded all datasets
 Target variable: Air Temperature

  PREPARING FEATURES
--------------------------------------------------------------------------------
  Using 50 features for modeling

 TRAINING MODELS AND CALCULATING METRICS
--------------------------------------------------------------------------------
  Training Linear Regression...


/Users/eekod/Library/Mobile Documents/com~apple~CloudDocs/UCSF MAS/DATASCI 217 Python/DATSCI217 Final/ds217-11-final-cshiu-1/venv/lib/python3.9/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/eekod/Library/Mobile Documents/com~apple~CloudDocs/UCSF MAS/DATASCI 217 Python/DATSCI217 Final/ds217-11-final-cshiu-1/venv/lib/python3.9/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/eekod/Library/Mobile Documents/com~apple~CloudDocs/UCSF MAS/DATASCI 217 Python/DATSCI217 Final/ds217-11-final-cshiu-1/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/eekod/Library/Mobile Documents/com~apple~CloudDocs/UCSF MAS/DATASCI 217 Python/DATSCI217 Final/ds217-11-final-cshiu-1/venv/lib/python3.9/site-packages/sklearn/linear_

  Training Random Forest...
  All models trained and evaluated

 GENERATING METRICS REPORT
--------------------------------------------------------------------------------
  Saved to: output/q7_model_metrics.txt
  Lines: 63
  File saved successfully
  Contains model names: True
  Contains R² scores: True
  Contains RMSE: True
  Contains MAE: True
  Contains train/test metrics: True

SUMMARY
  Generated metrics for 2 models
  Best model: Linear Regression (Test R²: 1.0000)
  Metrics included:
  - R² (R-squared) - REQUIRED ✓
  - RMSE (Root Mean Squared Error) - RECOMMENDED ✓
  - MAE (Mean Absolute Error) - RECOMMENDED ✓

  Report includes:
  - Individual model performance
  - Model comparison and ranking
  - Performance analysis
  - Overfitting detection

  Saved to: output/q7_model_metrics.txt

MODEL METRICS REPORT COMPLETE


### 3. `output/q7_feature_importance.csv`
**Format:** CSV file
**Required Columns (exact names):** `feature`, `importance`
**Content:** Feature importance from tree-based models (XGBoost, Random Forest)
**Requirements:**
- One row per feature
- `feature`: Feature name (string)
- `importance`: Importance score (float, typically 0-1, sum to 1)
- Sorted by importance (descending)
- **No index column** (save with `index=False`)

**Note:** Tree-based models (XGBoost, Random Forest) provide feature importance directly via `.feature_importances_`. If using only Linear Regression, you can use the absolute values of coefficients as a proxy for importance.

**Example:**
```csv
feature,importance
Air Temperature,0.6539
hour,0.1234
month,0.0892
Water Temperature,0.0456
...
```

In [4]:
# 7.3 Feature Importance csv

print("="*80)
print("Q7.3: EXTRACTING FEATURE IMPORTANCE")
print("="*80)

# Load train/test data
print("\n1. LOADING DATA")
print("-" * 80)

X_train = pd.read_csv('output/q6_X_train.csv')
y_train = pd.read_csv('output/q6_y_train.csv')

print(f"  X_train: {X_train.shape}")
print(f"  y_train: {y_train.shape}")

# Get target variable name
target_name = y_train.columns[0]
y_train_values = y_train[target_name].values

print(f"  Target variable: {target_name}")

# DATA LEAKAGE CHECK AND FEATURE SELECTION
print("\n PREPARING FEATURES (REPLICATING TRAINING PROCESS)")
print("-" * 80)

# Calculate correlations for leakage detection
feature_target_corr = pd.DataFrame({
    'feature': X_train.columns,
    'correlation': [X_train[col].corr(pd.Series(y_train_values)) for col in X_train.columns]
})
feature_target_corr['abs_correlation'] = feature_target_corr['correlation'].abs()
feature_target_corr = feature_target_corr.sort_values('abs_correlation', ascending=False)

# Remove leakage features
high_corr_features = feature_target_corr[feature_target_corr['abs_correlation'] > 0.95]
target_base = target_name.lower().replace(' ', '_').replace('_', '')
leakage_features = []

for feat in high_corr_features['feature']:
    feat_clean = feat.lower().replace(' ', '_').replace('_', '')
    if target_base in feat_clean and any(x in feat.lower() for x in ['rolling', 'lag', 'mean', 'std', 'deviation']):
        leakage_features.append(feat)

if leakage_features:
    X_train = X_train.drop(columns=leakage_features)
    print(f"Removed {len(leakage_features)} leakage features")

# Select top features
n_features = min(50, len(X_train.columns))
top_features = feature_target_corr.head(n_features)['feature'].tolist()
top_features = [f for f in top_features if f not in leakage_features]

X_train_selected = X_train[top_features].copy()

# Handle missing values
for col in X_train_selected.columns:
    if X_train_selected[col].isnull().any():
        mean_val = X_train_selected[col].mean()
        X_train_selected[col].fillna(mean_val, inplace=True)

print(f"  Using {len(top_features)} features for modeling")

# Train models and extract feature importance
print("\n TRAINING MODELS AND EXTRACTING FEATURE IMPORTANCE")
print("-" * 80)

# Define models
models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
}

feature_importances = {}

# Train tree-based models and extract importances
for model_name, model in models.items():
    print(f"\nTraining {model_name}...")
    model.fit(X_train_selected, y_train_values)
    
    # Get feature importances
    importances = model.feature_importances_
    
    # Store importances
    feature_importances[model_name] = pd.DataFrame({
        'feature': X_train_selected.columns,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    print(f"  Extracted feature importances")
    print(f"  Top 3 features:")
    for i, row in feature_importances[model_name].head(3).iterrows():
        print(f"    {row['feature']}: {row['importance']:.4f}")

# Also get Linear Regression coefficients as proxy for importance
print(f"\nTraining Linear Regression (for comparison)...")
lr_model = LinearRegression()
lr_model.fit(X_train_selected, y_train_values)

# Use absolute values of coefficients as importance proxy
lr_coefs = np.abs(lr_model.coef_)
# Normalize to sum to 1 (like tree importances)
lr_coefs_normalized = lr_coefs / lr_coefs.sum()

feature_importances['Linear Regression'] = pd.DataFrame({
    'feature': X_train_selected.columns,
    'importance': lr_coefs_normalized
}).sort_values('importance', ascending=False)

print(f"  Extracted coefficient importances")

# Average importance across all tree-based models
print("\n CALCULATING AVERAGE FEATURE IMPORTANCE")
print("-" * 80)

# Use only tree-based models for the main importance file
tree_models = ['Random Forest']

# Calculate average importance across tree models
avg_importance = pd.DataFrame({'feature': X_train_selected.columns})
for model_name in tree_models:
    avg_importance = avg_importance.merge(
        feature_importances[model_name][['feature', 'importance']].rename(
            columns={'importance': f'importance_{model_name.replace(" ", "_").lower()}'}
        ),
        on='feature'
    )

# Calculate mean importance
importance_cols = [col for col in avg_importance.columns if col.startswith('importance_')]
avg_importance['importance'] = avg_importance[importance_cols].mean(axis=1)

# Keep only feature and importance columns, sort by importance
final_importance = avg_importance[['feature', 'importance']].sort_values('importance', ascending=False)

print(f" Calculated average importance across {len(tree_models)} tree-based models")
print(f"\nTop 10 most important features:")
for i, row in final_importance.head(10).iterrows():
    print(f"  {i+1:2d}. {row['feature']}: {row['importance']:.4f}")

# Verify importance sums to approximately 1
total_importance = final_importance['importance'].sum()
print(f"\nTotal importance: {total_importance:.4f} (should be ~1.0)")

# Save feature importance
print("\n  SAVING FEATURE IMPORTANCE")
print("-" * 80)

final_importance.to_csv('output/q7_feature_importance.csv', index=False)

print(f"  Saved to: output/q7_feature_importance.csv")
print(f"  Shape: {final_importance.shape[0]} features × 2 columns")
print(f"  Sorted: Descending by importance")
print(f"  Format: feature (string), importance (float)")


# Summary
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"  Extracted feature importance from {len(tree_models)} tree-based models")
print(f"  Number of features: {len(final_importance)}")
print(f"  Top 3 most important features:")
for i, row in final_importance.head(3).iterrows():
    print(f"    {i+1}. {row['feature']}: {row['importance']:.4f}")

print(f"\n  - Most important feature explains: {final_importance.iloc[0]['importance']*100:.1f}% of variance")

print(f"\n  Output files:")
print(f"  - output/q7_feature_importance.csv")

print("\n" + "="*80)
print("FEATURE IMPORTANCE EXTRACTION COMPLETE")
print("="*80)


Q7.3: EXTRACTING FEATURE IMPORTANCE

1. LOADING DATA
--------------------------------------------------------------------------------
  X_train: (157237, 88)
  y_train: (157237, 1)
  Target variable: Air Temperature

 PREPARING FEATURES (REPLICATING TRAINING PROCESS)
--------------------------------------------------------------------------------
  Using 50 features for modeling

 TRAINING MODELS AND EXTRACTING FEATURE IMPORTANCE
--------------------------------------------------------------------------------

Training Random Forest...


/Users/eekod/Library/Mobile Documents/com~apple~CloudDocs/UCSF MAS/DATASCI 217 Python/DATSCI217 Final/ds217-11-final-cshiu-1/venv/lib/python3.9/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/eekod/Library/Mobile Documents/com~apple~CloudDocs/UCSF MAS/DATASCI 217 Python/DATSCI217 Final/ds217-11-final-cshiu-1/venv/lib/python3.9/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


  Extracted feature importances
  Top 3 features:
    Wet Bulb Temperature: 0.9575
    Air Temperature_change_24h: 0.0142
    Air Temperature_lag_24h: 0.0119

Training Linear Regression (for comparison)...
  Extracted coefficient importances

 CALCULATING AVERAGE FEATURE IMPORTANCE
--------------------------------------------------------------------------------
 Calculated average importance across 1 tree-based models

Top 10 most important features:
   1. Wet Bulb Temperature: 0.9575
  27. Air Temperature_change_24h: 0.0142
   3. Air Temperature_lag_24h: 0.0119
  47. Humidity_change_24h: 0.0076
  45. Humidity_lag_24h: 0.0065
  44. Battery Life: 0.0017
  38. Humidity_rolling_mean_168h: 0.0001
   9. Total Rain_rolling_mean_168h: 0.0000
  17. Maximum Wind Speed_rolling_std_168h: 0.0000
  48. Humidity_rolling_std_168h: 0.0000

Total importance: 1.0000 (should be ~1.0)

  SAVING FEATURE IMPORTANCE
--------------------------------------------------------------------------------
  Saved to: 